In [0]:
display(dbutils.fs.ls("/Volumes/pharmacy_sales/silver/silver_layer/silver_vw_sales.parquet"))

path,name,size,modificationTime
dbfs:/Volumes/pharmacy_sales/silver/silver_layer/silver_vw_sales.parquet,silver_vw_sales.parquet,151255910,1788979543000


In [0]:
df = spark.read.parquet("/Volumes/pharmacy_sales/silver/silver_layer/silver_vw_sales.parquet")

In [0]:
# Use PyArrow to handle nanosecond timestamp, then convert to microseconds
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# Read with PyArrow
table = pq.read_table("/Volumes/pharmacy_sales/silver/silver_layer/silver_vw_sales.parquet")

# Convert any nanosecond timestamp columns to microsecond precision (Spark-compatible)
for i, field in enumerate(table.schema):
    if pa.types.is_timestamp(field.type) and field.type.unit == 'ns':
        col = table.column(i)
        col_micro = pc.cast(col, pa.timestamp('us'))
        table = table.set_column(i, field.name, col_micro)

# Convert to Spark DataFrame
df = spark.createDataFrame(table.to_pandas())
print("Rows: ",df.count())
display(df.limit(20))

Rows:  7176775


store_id,cust_group_id,item_id,item_name,category,sub_category,pack_size,generic_flag,bill_no,bill_no_source_type,bill_datetime,bill_date,bill_month_start,sale_qty,sale_value,discount_amt,gross_sale_value,is_return,bill_count,max_sale_qty,offer_flag
14775,5991,TRA0866,TRAVOPURE 0.004%W/V EYE DROPS 3ML,PHARMA,EYE DROPS,1,null,101359192,NUMERIC,2024-02-17T18:09:00.000Z,2024-02-17,2024-02-01,-1,-350.0,null,-350.0,1,1,0,null
14775,5153,DAL0004,DALACIN C 300MG CAP 10'S,PHARMA,CAPSULE,10,null,101361864,NUMERIC,2024-03-28T18:36:00.000Z,2024-03-28,2024-03-01,-10,-297.6000061035156,-52.08000183105469,-349.68,1,1,0,null
14775,102,BOL0044,BOLD SECRET FIRE BODY SPRAY 200ML,FMCG,SPRAY,1,null,101365932,NUMERIC,2024-06-06T21:51:00.000Z,2024-06-06,2024-06-01,-1,-399.0,-199.5,-598.5,1,1,0,OFFER
14775,7661,HIM0246,HIMALAYA ANTI-HAIR FALL BHRINGARAJA SMP 180ML,FMCG,PERSONAL CARE,1,null,101366705,NUMERIC,2024-06-21T15:37:00.000Z,2024-06-21,2024-06-01,-1,-160.0,0.0,-160.0,1,1,0,null
14775,102,VIC0024,VICKS VAPORUB 25ML,FMCG,OTC,1,null,101367250,NUMERIC,2024-06-30T17:49:00.000Z,2024-06-30,2024-06-01,-1,-109.0,0.0,-109.0,1,1,0,null
14775,102,DYT0004,DYTOR 10MG TAB 15'S,PHARMA,TABLET,15,null,101370754,NUMERIC,2024-08-23T21:43:00.000Z,2024-08-23,2024-08-01,-30,-205.1999969482422,0.0,-205.2,1,1,0,null
14775,102,URI0255,URIMAX 0.4MG MR CAP 20'S,PHARMA,CAPSULE,20,null,101370754,NUMERIC,2024-08-23T21:43:00.000Z,2024-08-23,2024-08-01,-40,-629.2000122070312,0.0,-629.2,1,1,0,null
14775,7661,DEX0296,DEXOLAC STAGE-1 (0-6M) INFANT FORMULA POW 400G REFILL,FMCG,BABY CARE,1,null,101372965,NUMERIC,2024-09-30T11:18:00.000Z,2024-09-30,2024-09-01,-1,-460.0,0.0,-460.0,1,1,0,null
14775,0,FIN0063,FINATE 160MG TAB 10'S,PHARMA,TABLET,10,null,101375818,NUMERIC,2024-11-11T22:59:00.000Z,2024-11-11,2024-11-01,-30,-499.20001220703125,0.0,-499.2,1,1,0,null
14775,0,FIN0063,FINATE 160MG TAB 10'S,PHARMA,TABLET,10,null,101376021,NUMERIC,2024-11-15T18:02:00.000Z,2024-11-15,2024-11-01,-30,-499.20001220703125,0.0,-499.2,1,1,0,null


In [0]:
df.printSchema()
df.write.mode("overwrite").saveAsTable("silver_vw_sales")

root
 |-- store_id: long (nullable = true)
 |-- cust_group_id: long (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- pack_size: long (nullable = true)
 |-- generic_flag: string (nullable = true)
 |-- bill_no: string (nullable = true)
 |-- bill_no_source_type: string (nullable = true)
 |-- bill_datetime: timestamp (nullable = true)
 |-- bill_date: date (nullable = true)
 |-- bill_month_start: date (nullable = true)
 |-- sale_qty: long (nullable = true)
 |-- sale_value: double (nullable = true)
 |-- discount_amt: double (nullable = true)
 |-- gross_sale_value: double (nullable = true)
 |-- is_return: long (nullable = true)
 |-- bill_count: long (nullable = true)
 |-- max_sale_qty: long (nullable = true)
 |-- offer_flag: string (nullable = true)

